# Data Split

Splits the main dataset's processed data into a stratified training and
test set and saves them under the dataset's `final_data/` folder (paths
and the split ratio come from `configs/config.json`). Feature selection
and the models then work on the training file only, so no information
leaks from the test data into training.

This notebook's job is producing those files - it is the pipeline's
dedicated write step.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

root = Path.cwd()
while not (root / "configs" / "config.json").is_file():
    if root == root.parent:
        raise FileNotFoundError("configs/config.json not found above the working directory")
    root = root.parent

config = json.loads((root / "configs" / "config.json").read_text())


def dataset_file(ds_id, stage):
    ds = config["datasets"]["list"][ds_id]
    entry = ds["files"][stage]
    if isinstance(entry, dict):
        entry = entry.get("train", entry.get("data"))
    elif isinstance(entry, list):
        entry = entry[0]
    return root / "configs" / config["paths"]["datasets_dir"] / ds["folder"] / entry

In [ ]:
ds_id = config["datasets"]["main_dataset"]
ds = config["datasets"]["list"][ds_id]
df = pd.read_csv(dataset_file(ds_id, "processed"), na_values=ds["na_values"] or None)
target = ds["target"]["column"]

train, test = train_test_split(
    df,
    test_size=config["training"]["test_size"],
    random_state=config["training"]["random_state"],
    stratify=df[target],
)

train.to_csv(dataset_file(ds_id, "final"), index=False)
test.to_csv(root / "configs" / config["paths"]["datasets_dir"] / ds["folder"] / ds["files"]["final"]["test"], index=False)

print(f"{ds_id} - {ds['name']}: split {len(df)} rows into "
      f"{len(train)} train / {len(test)} test "
      f"(test_size = {config['training']['test_size']}, seed = {config['training']['random_state']})")
balance = pd.DataFrame({
    "train %": (train[target].value_counts(normalize=True) * 100).round(2),
    "test %": (test[target].value_counts(normalize=True) * 100).round(2),
}).sort_index()
print(balance.to_string())
print(f"saved: {ds['files']['final']['train']} , {ds['files']['final']['test']}")